<a href="https://colab.research.google.com/github/noobylub/Dissertation_Project/blob/master/test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Check what device I am running on, and I want to make sure I am running on TPU
import torch

def check_gpu():
    """Check GPU availability"""
    if torch.cuda.is_available():
        print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
        return torch.device("cuda")


    else:
        print("📱 Using CPU")
        return torch.device("cpu")

device = check_gpu()

✅ GPU: Tesla T4


In [2]:
# If running on google colab website
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

In [ ]:
from dotenv import load_dotenv
import os

# Load the .env file (adjust path if needed)
load_dotenv('.env')

# Now access the variables
hf_token = os.getenv('HF_TOKEN')
print(hf_token)

In [3]:
# Install bitsandbytes for quantisation
!pip install -U bitsandbytes>=0.46.1

Use these models: https://huggingface.co/google/gemma-7b
<br/>
Second model to test: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct

In [ ]:
# Diagnostic check
import sys
print(f"Python version: {sys.version}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")

# Check if bitsandbytes can be imported
try:
    import bitsandbytes as bnb
    print(f"✅ bitsandbytes imported successfully: {bnb.__version__}")
except ImportError as e:
    print(f"❌ Failed to import bitsandbytes: {e}")

# Check if transformers can detect bitsandbytes
try:
    from transformers.utils.quantization_config import is_bitsandbytes_available
    print(f"is_bitsandbytes_available(): {is_bitsandbytes_available()}")
except Exception as e:
    print(f"Error checking bitsandbytes availability: {e}")

# Check if bitsandbytes has CUDA support
try:
    from bitsandbytes.cextension import COMPILED_WITH_CUDA
    print(f"bitsandbytes compiled with CUDA: {COMPILED_WITH_CUDA}")
except Exception as e:
    print(f"Error checking CUDA compilation: {e}")

In [4]:
# Lading the transformer models into the GPU

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# from dotenv import load_dotenv
# import os

# # Load the .env file (adjust path if needed)
# load_dotenv('.env')

# # Now access the variables
# hf_token = os.getenv('HF_TOKEN')

# Quantisation
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

# Trying to run inference on Llama 3.1 model
# Add device mapping and quantization
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=hf_token,
    padding_side='left',


)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=hf_token,
    device_map="auto",           # Auto device placement
    quantization_config=bnb_config
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:
import torch

def get_model_size(model):
    param_size = 0
    buffer_size = 0

    for param in model.parameters():
        param_size += param.nelement() * param.element_size()

    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()

    total_size = (param_size + buffer_size) / 1024**3  # GB

    print(f"Parameters: {param_size / 1024**3:.2f} GB")
    print(f"Buffers: {buffer_size / 1024**3:.2f} GB")
    print(f"Total: {total_size:.2f} GB")

    return total_size

# Check your model size
model_size_gb = get_model_size(model)

Parameters: 8.46 GB
Buffers: 0.00 GB
Total: 8.46 GB


In [ ]:
def hook_extract_vector(module, input, output):
    if isinstance(output, tuple):
        hidden_state = output[0]
    else:
        hidden_state = output  # Fixed: was output.last_hidden_state

    extracted_activations.append(hidden_state.detach().cpu())
    return output

print("Hook function redefined!")

Hook function redefined!


In [ ]:
model.model._forward_hooks.clear()

In [9]:
# This is for extracting vectors
# We mostly follow this method: https://elib.dlr.de/218629/1/The_Effectiveness_of_Style_Vectors_for_Steering_Large_Language_Models_A_Human_Evaluation.pdf
# Extracting the input representation and averaging them to determine layer representation

import torch


# model generates with the given prompt and extracts the vectors
def generate_extract(user_text: str, model, tokenizer, max_new_tokens=200, layer_idx=15):

    messages = [
        {"role": "user", "content": user_text},

    ]

    encoded_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)


    with torch.no_grad():
        outputs = model(
            **encoded_inputs,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            output_hidden_states = True,
        )
        # output = model(**inputs)
        # outputs = output

    # input_length = input_ids.shape[1]
    # generated_text = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    # print(generated_text)

    # extract_hook.remove()

    return outputs.hidden_states

In [ ]:
# This would be for steering
import torch

# Code set up beforehand

# Global storage (define outside functions)
# extracted_activations = []

# Method to extract vectors from model
def extract_vector(module, input, output,vector_steer,strength):
    # The output from model.model (LlamaModel) during generation is typically a BaseModelOutputWithPast object.
    # We want to extract the last_hidden_state tensor from it.
    if isinstance(output, tuple):
        hidden_state = output[0]
    else:
        hidden_state = output

    # Detach the tensor from the graph and move to CPU to save memory if not needed for backprop
    # extracted_activations.append(hidden_state.detach().cpu())

    # IMPORTANT: Hooks should return the original output or a modified one. If only observing, return original output.
    # If steerign you can modify it first and return that
    return hidden_state + (vector_steer * strength )

# model generates with the given prompt and extracts the vectors
def generate_steer(user_text: str, system_text:str, model, tokenizer, max_new_tokens=200, layer_idx=15):
    extracted_activations.clear()

    extract_hook = model.model.layers[layer_idx].register_forward_hook(extract_vector)
    print("Attached to layer", layer_idx)

    messages = [
        {"role": "user", "content": user_text},
        {"role":"system", "content":system_text}
    ]

    encoded_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    input_ids = encoded_inputs["input_ids"]
    attention_mask = encoded_inputs["attention_mask"]

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_length = input_ids.shape[1]
    generated_text = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    print(generated_text)

    extract_hook.remove()

    return generated_text, extracted_activations.copy()

In [21]:
# Loading dataset
!wget -P data/full_dataset/ https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_1.csv

--2026-03-16 00:16:34--  https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_1.csv
Resolving storage.googleapis.com (storage.googleapis.com)... 142.250.101.207, 142.250.141.207, 142.251.2.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.250.101.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14174600 (14M) [application/octet-stream]
Saving to: ‘data/full_dataset/goemotions_1.csv’

goemotions_1.csv    100%[===================>]  13.52M  62.8MB/s    in 0.2s    

2026-03-16 00:16:34 (62.8 MB/s) - ‘data/full_dataset/goemotions_1.csv’ saved [14174600/14174600]



In [14]:
# Prompts designed to ellicity joy emotion
prompt = "in more than 100 words describe how your day is going"


In [19]:
generated = generate_extract(prompt, model, tokenizer, max_new_tokens=200, layer_idx=13)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


In [20]:

vector = (generated)
# print(text)
for i in range(len(vector)):
  print(vector[i].shape)



torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])


In [ ]:
print(vectors2[1].shape)

torch.Size([1, 1, 4096])
